# Comparing Experiments

This notebook compares results across different experiments.

In [1]:
from pathlib import Path
import pandas as pd

### Averaging Across Seeds
*Experiments: 10–13*  
This section averages the ResNet-18 candidate experiments, which were run across four different seeds.

In [2]:
def load_results(label):
    """
    Load results for a given label from the parquet file for experiments 10–13.

    Args:
        label (str): The label for which to load results.

    Returns:
        pd.DataFrame: A DataFrame containing the loaded results.
    """

    label = label + "_classify"
    df = pd.read_parquet(Path("../results") / label / "experiments.parquet")
    df = df[df["experiment_number"].isin([10, 11, 12, 13])]
    return df

In [3]:
# Columns to average across seeds
METRIC_COLUMNS = [
    "test_acc",
    "test_weighted_f1_avg",
    "test_unseen_matched_acc",
    "test_unseen_matched_weighted_f1_avg",
]


def average_seed_results(results):
    """
    Return mean and standard deviation for each candidate across seeds, rounded to 2 DP.

    Args:
        results (pd.DataFrame): A DataFrame containing the results to be averaged.

    Returns:
        pd.DataFrame: A DataFrame containing the mean and standard deviation for each
            candidate across seeds.
    """

    seed_counts = results.groupby("run_name")["seed"].nunique()
    summary = results.groupby("run_name")[METRIC_COLUMNS].agg(["mean", "std"])
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    summary.insert(0, "n_seeds", seed_counts)
    return summary.reset_index().round(2)


def export_seed_comparison(label):
    """
    Average seed results, export them to CSV, and return the summary.

    Args:
        label (str): The label for which to export seed comparison.

    Returns:
        pd.DataFrame: A DataFrame containing the averaged seed results.
    """

    summary = average_seed_results(load_results(label))
    filename = "candidate_seed_comparison"

    output_dir = Path("../acc_f1_tables") / f"{label}_classify"
    output_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(output_dir / f"{filename}.csv", index=False)
    return summary

#### Object

In [4]:
export_seed_comparison("object")

,run_name,n_seeds,test_acc_mean,test_acc_std,test_weighted_f1_avg_mean,test_weighted_f1_avg_std,test_unseen_matched_acc_mean,test_unseen_matched_acc_std,test_unseen_matched_weighted_f1_avg_mean,test_unseen_matched_weighted_f1_avg_std
0,crop_all,4,86.29,2.38,0.86,0.02,56.19,2.23,0.63,0.02
1,pad_flip_jitter,4,91.66,0.89,0.92,0.01,50.61,1.03,0.56,0.02
2,pad_jitter,4,93.28,0.97,0.93,0.01,51.16,2.15,0.55,0.04
3,pad_none,4,93.86,1.20,0.94,0.01,46.66,2.89,0.51,0.03


#### Object Region

In [5]:
export_seed_comparison("object_region")

,run_name,n_seeds,test_acc_mean,test_acc_std,test_weighted_f1_avg_mean,test_weighted_f1_avg_std,test_unseen_matched_acc_mean,test_unseen_matched_acc_std,test_unseen_matched_weighted_f1_avg_mean,test_unseen_matched_weighted_f1_avg_std
0,crop_all,4,80.75,1.77,0.80,0.02,39.63,1.86,0.45,0.01
1,pad_flip_jitter,4,87.29,1.10,0.87,0.01,34.67,2.69,0.38,0.01
2,pad_jitter,4,87.25,3.21,0.87,0.03,32.33,1.47,0.34,0.01
3,pad_none,4,91.23,1.64,0.91,0.02,29.79,1.26,0.30,0.02
